### Byte pair algorithm example

In [ ]:
import tiktoken

In [ ]:
with open("the-verdict.txt","r") as f:
    text=f.read()

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2") 
text="Hello world <|endoftext|> "
tokenizer.encode(text,allowed_special={"<|endoftext|>"})

In [ ]:
tokenizer.decode([15496, 995, 220, 50256, 220])

### Byte pair algorithm implementation

In [ ]:
text="""The Internet is a global system that connects computers and devices worldwide, enabling communication, information sharing, and access to digital services.

Connects people and devices globally
Enables communication, learning, and business
Supports digital services and cloud platforms
Operates using standard communication protocols
Forms the backbone of modern digital life """

In [ ]:
encode=list(text.encode(encoding="utf-8"))

In [ ]:
encode

In [ ]:
dict={char:encode.count(char)for char in encode}  ## freq, count of each character
sorted(dict.items(),key=lambda x:x[1],reverse=True) ##  character, freq format

In [ ]:
pair=[]
for p in zip(encode,encode[1:]):
    pair.append(p)

In [ ]:
pair_count={p:pair.count(p) for p in pair}

In [ ]:
pair_count

In [ ]:
del dict
sorted_pair=dict(sorted(pair_count.items(),key=lambda x:x[1],reverse=True))
sorted_pair

In [ ]:
top_pair=max(sorted_pair,key=sorted_pair.get)
top_pair

In [ ]:
def merge_id(ids:list,pair_seq, id:int)->list:
    """ids : list, Old list containing ids of each byte in text
       pair_seq : tuple, Sequence we want to capture through breakpoint
       id : int, id we want to replce when we encounter the sequence
    """
    merge={}
    new_id=[]
    leng=len(ids)
    i=0
    while i < leng:
            
            if(i<leng-1 and ids[i]==pair_seq[0] and ids[i+1] == pair_seq[1]):
                new_id.append(id)
                i=i+2
            else:
                 new_id.append(ids[i]) 
                 i=i+1     
    return new_id       

In [ ]:
new_id=merge_id(encode,top_pair,max(encode)+1)

In [ ]:
new_id

In [ ]:
def encode_seq(text,epocs):
    """
    epocs:-> How many seq to merges; based on no of occurance
    """
    encode=list(text.encode(encoding="utf-8"))

    merge={}

    for i in range(epocs):
        pair=[] ## find pairs of each consicutive pair
        for p in zip(encode,encode[1:]):
            pair.append(p)
        pair_count={p:pair.count(p) for p in pair}
        sorted_pair=dict(sorted(pair_count.items(),key=lambda x:x[1],reverse=True))
        top_pair=max(sorted_pair,key=sorted_pair.get) 
        merge_by=max(encode)+1 
        encode=merge_id(encode,top_pair,merge_by)
        print(f" Merging {top_pair}->{merge_by}")  
        if merge_by not in merge:
            merge[merge_by]=[]
            merge[merge_by].append(top_pair[0])
            merge[merge_by].append(top_pair[1])
        
            
    return encode,merge

In [ ]:
# with open("the-verdict.txt","r",encoding="utf-8") as file:
#     text=file.read()

In [ ]:
# len(encode)-len(final_encode)

In [ ]:
def recursive_check(merge, id):
    """
    When the merge is obtained we might obtain some double merges or merges inside merges, to resolve that we use this function
    """
    result = []

    for x in merge[id]:

        if x in merge:
            result.extend(recursive_check(merge, x))
        else:
            result.append(x)


    return result

In [ ]:
def decode_seq(ids: list,merge) -> str:
    x = ""

    for id in ids:
        vals=recursive_check(merge,id)
        x = "".join(bytes([v]).decode("utf-8", errors="replace")
        for v in vals)
    return x

In [ ]:
text=text[:50]
encode,merge=encode_seq(text,4)
decode_seq(encode,merge)